In [1]:
print('hello')

hello


In [2]:
import pandas as pd
import math

docs = [
    "Battery life is great camera quality is good",
    "Battery drains fast phone heats while gaming",
    "Camera is excellent in low light battery is decent",
    "Phone design is good performance is smooth"
]

doc_ids = ["D1", "D2", "D3", "D4"]

pd.DataFrame({"doc_id": doc_ids, "text": docs})


,doc_id,text
0,D1,Battery life is great camera quality is good
1,D2,Battery drains fast phone heats while gaming
2,D3,Camera is excellent in low light battery is de...
3,D4,Phone design is good performance is smooth


In [3]:
# Lowercase + split all docs into words
tokenized_docs = [doc.lower().replace(",", "").split() for doc in docs]
tokenized_docs


[['battery', 'life', 'is', 'great', 'camera', 'quality', 'is', 'good'],
 ['battery', 'drains', 'fast', 'phone', 'heats', 'while', 'gaming'],
 ['camera',
  'is',
  'excellent',
  'in',
  'low',
  'light',
  'battery',
  'is',
  'decent'],
 ['phone', 'design', 'is', 'good', 'performance', 'is', 'smooth']]

In [4]:
# Get all unique words across all docs
vocab = sorted(set(word for doc in tokenized_docs for word in doc))
vocab


['battery',
 'camera',
 'decent',
 'design',
 'drains',
 'excellent',
 'fast',
 'gaming',
 'good',
 'great',
 'heats',
 'in',
 'is',
 'life',
 'light',
 'low',
 'performance',
 'phone',
 'quality',
 'smooth',
 'while']

In [5]:
# Initialize empty TF matrix (list of dicts)
tf_rows = []

for doc_tokens, doc_id in zip(tokenized_docs, doc_ids):
    row = {}
    for word in vocab:
        row[word] = doc_tokens.count(word)
    row["doc_id"] = doc_id
    tf_rows.append(row)

# Convert to DataFrame
df_TF = pd.DataFrame(tf_rows).set_index("doc_id")
df_TF


,battery,camera,decent,design,drains,excellent,fast,gaming,good,great,...,in,is,life,light,low,performance,phone,quality,smooth,while
doc_id,,,,,,,,,,,,,,,,,,,,,
D1,1,1,0,0,0,0,0,0,1,1,...,0,2,1,0,0,0,0,1,0,0
D2,1,0,0,0,1,0,1,1,0,0,...,0,0,0,0,0,0,1,0,0,1
D3,1,1,1,0,0,1,0,0,0,0,...,1,2,0,1,1,0,0,0,0,0
D4,0,0,0,1,0,0,0,0,1,0,...,0,2,0,0,0,1,1,0,1,0


In [6]:
df_dict = {}

for word in vocab:
    # count how many docs have this word with frequency > 0
    df_count = (df_TF[word] > 0).sum()
    df_dict[word] = df_count

df_DF = pd.DataFrame.from_dict(df_dict, orient="index", columns=["DF"])
df_DF


,DF
battery,3
camera,2
decent,1
design,1
drains,1
excellent,1
fast,1
gaming,1
good,2
great,1


In [7]:
N = len(doc_ids)

idf_dict = {}
for word, df_val in df_dict.items():
    idf_dict[word] = math.log(N / df_val)

df_IDF = pd.DataFrame.from_dict(idf_dict, orient="index", columns=["IDF"])
df_IDF


,IDF
battery,0.287682
camera,0.693147
decent,1.386294
design,1.386294
drains,1.386294
excellent,1.386294
fast,1.386294
gaming,1.386294
good,0.693147
great,1.386294


In [8]:
# Create empty TF-IDF matrix with same shape as TF
df_TFIDF = df_TF.copy().astype(float)

for word in vocab:
    df_TFIDF[word] = df_TF[word] * df_IDF.loc[word, "IDF"]

df_TFIDF


,battery,camera,decent,design,drains,excellent,fast,gaming,good,great,...,in,is,life,light,low,performance,phone,quality,smooth,while
doc_id,,,,,,,,,,,,,,,,,,,,,
D1,0.287682,0.693147,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.693147,1.386294,...,0.000000,0.575364,1.386294,0.000000,0.000000,0.000000,0.000000,1.386294,0.000000,0.000000
D2,0.287682,0.000000,0.000000,0.000000,1.386294,0.000000,1.386294,1.386294,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.693147,0.000000,0.000000,1.386294
D3,0.287682,0.693147,1.386294,0.000000,0.000000,1.386294,0.000000,0.000000,0.000000,0.000000,...,1.386294,0.575364,0.000000,1.386294,1.386294,0.000000,0.000000,0.000000,0.000000,0.000000
D4,0.000000,0.000000,0.000000,1.386294,0.000000,0.000000,0.000000,0.000000,0.693147,0.000000,...,0.000000,0.575364,0.000000,0.000000,0.000000,1.386294,0.693147,0.000000,1.386294,0.000000


In [9]:
df_TF.round(0), df_DF.T, df_IDF.T, df_TFIDF.round(3)


(        battery  camera  decent  design  drains  excellent  fast  gaming  \
 doc_id                                                                     
 D1            1       1       0       0       0          0     0       0   
 D2            1       0       0       0       1          0     1       1   
 D3            1       1       1       0       0          1     0       0   
 D4            0       0       0       1       0          0     0       0   
 
         good  great  ...  in  is  life  light  low  performance  phone  \
 doc_id               ...                                                 
 D1         1      1  ...   0   2     1      0    0            0      0   
 D2         0      0  ...   0   0     0      0    0            0      1   
 D3         0      0  ...   1   2     0      1    1            0      0   
 D4         1      0  ...   0   2     0      0    0            1      1   
 
         quality  smooth  while  
 doc_id                          
 D1            1

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(lowercase=True, stop_words='english')
tfidf_matrix = vectorizer.fit_transform(docs)

df_sklearn = pd.DataFrame(
    tfidf_matrix.toarray(),
    index=doc_ids,
    columns=vectorizer.get_feature_names_out()
)

df_sklearn.round(3)



,battery,camera,decent,design,drains,excellent,fast,gaming,good,great,heats,life,light,low,performance,phone,quality,smooth
D1,0.296,0.366,0.000,0.000,0.000,0.000,0.000,0.000,0.366,0.464,0.000,0.464,0.000,0.000,0.000,0.000,0.464,0.000
D2,0.285,0.000,0.000,0.000,0.446,0.000,0.446,0.446,0.000,0.000,0.446,0.000,0.000,0.000,0.000,0.352,0.000,0.000
D3,0.285,0.352,0.446,0.000,0.000,0.446,0.000,0.000,0.000,0.000,0.000,0.000,0.446,0.446,0.000,0.000,0.000,0.000
D4,0.000,0.000,0.000,0.485,0.000,0.000,0.000,0.000,0.383,0.000,0.000,0.000,0.000,0.000,0.485,0.383,0.000,0.485


In [24]:
df_sklearn.loc['D1']

battery        0.295980
camera         0.365594
decent         0.000000
design         0.000000
drains         0.000000
excellent      0.000000
fast           0.000000
gaming         0.000000
good           0.365594
great          0.463709
heats          0.000000
life           0.463709
light          0.000000
low            0.000000
performance    0.000000
phone          0.000000
quality        0.463709
smooth         0.000000
Name: D1, dtype: float64

In [22]:
df_TFIDF.loc['D1']

battery        0.287682
camera         0.693147
decent         0.000000
design         0.000000
drains         0.000000
excellent      0.000000
fast           0.000000
gaming         0.000000
good           0.693147
great          1.386294
heats          0.000000
in             0.000000
is             0.575364
life           1.386294
light          0.000000
low            0.000000
performance    0.000000
phone          0.000000
quality        1.386294
smooth         0.000000
while          0.000000
Name: D1, dtype: float64

In [25]:
from sklearn.metrics.pairwise import cosine_similarity

# df_TFIDF is your TF-IDF matrix (Docs × Terms)
similarity_matrix = cosine_similarity(df_TFIDF)

pd.DataFrame(similarity_matrix, index=doc_ids, columns=doc_ids)


,D1,D2,D3,D4
D1,1.000000,0.009711,0.103263,0.114317
D2,0.009711,1.000000,0.008007,0.056705
D3,0.103263,0.008007,1.000000,0.038450
D4,0.114317,0.056705,0.038450,1.000000


In [27]:
df_TFIDF.index

Index(['D1', 'D2', 'D3', 'D4'], dtype='object', name='doc_id')

In [26]:
for i in df_TFIDF.index:
    print(df_TFIDF.loc(i))

ValueError: No axis named D1 for object type DataFrame